# DeepMTL2R Feature Gating Runner (Kaggle)

Notebook ini menjalankan eksperimen **Dynamic Feature Gating (Soft Gating)** menggunakan `run_extension.py`, dengan **pendekatan fully dynamic** untuk mendukung berbagai jumlah tasks.

## Konfigurasi (Dynamic Approach - End-to-End)

Notebook + run_extension.py menggunakan **pendekatan dinamis** untuk mendukung jumlah tasks yang bervariasi:

- `TASK_INDICES = [0, 131, 132, 133, 134, 135]` — Dapat diubah ke jumlah tasks **apapun**
- `N_TASKS = len(TASK_INDICES)` — Otomatis dihitung dari task indices
- `LABEL_INDICES = TASK_INDICES[1:]` — Auxiliary tasks, dihitung dinamis
- **Notebook validation**: Memvalidasi weights dari `weights-5tasks.txt` sebelum training
- **run_extension.py (UPDATED)**: 
  - `get_weight_combinations()` kini support arbitrary num_tasks
  - Ambil **first N_TASKS weights** dari setiap line di weights-5tasks.txt
  - Handle case untuk 2 tasks, 6 tasks, dan N tasks secara umum
  
**Keuntungan:**
- Ganti task indices → semua konfigurasi otomatis adjust
- Tidak perlu ubah jumlah weights secara manual
- Validation dilakukan di 2 level: notebook dan run_extension.py
- Support any number of tasks selama weights file punya cukup columns

## Output

- Semua metrik per-fold dan agregasi mean±std
- `gating_summary.json`
- **Gating Sparsity Ratio** (metrik khusus Feature Gating per epoch)
- **Robustness to Noisy Features** (ketahanan model terhadap noise)
- Plotting perbandingan metrik
- Tabel ringkasan hasil dengan visualisasi komprehensif

In [ ]:
!rm -rf /kaggle/working/DeepMTL2R
!git clone https://github.com/jteo0/DeepMTL2R.git

In [ ]:
import os
import sys
import shutil
import subprocess
from contextlib import redirect_stdout, redirect_stderr

with open(os.devnull, 'w') as fnull:
    with redirect_stdout(fnull), redirect_stderr(fnull):

        # Install dependencies
        subprocess.run(
            ['pip', 'install', '-r', '/kaggle/working/DeepMTL2R/requirements.txt'],
            check=False
        )

        # Install editable package
        subprocess.run(
            ['pip', 'install', '-e', '/kaggle/working/DeepMTL2R'],
            check=False
        )

        # Copy dataset
        INPUT_DS = '/kaggle/input/mslr-web10k/MSLR-WEB10K'
        TARGET_DS = '/kaggle/working/DeepMTL2R/datasets/MSLR-WEB10K'

        if os.path.exists(INPUT_DS):
            os.makedirs(TARGET_DS, exist_ok=True)
            shutil.copytree(INPUT_DS, TARGET_DS, dirs_exist_ok=True)

        # Update PYTHONPATH
        if '/kaggle/working/DeepMTL2R' not in sys.path:
            sys.path.insert(0, '/kaggle/working/DeepMTL2R')

print("Setup complete.")

In [ ]:
import gc
import json
import random
from collections import defaultdict
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch

plt.style.use('seaborn-v0_8-whitegrid')
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())

In [ ]:
# ==============================
# Runtime Config (Notebook-owned)
# ==============================

PROJECT_ROOT = '/kaggle/working/DeepMTL2R'
EXAMPLE_DIR = os.path.join(PROJECT_ROOT, 'examples', 'MSLR-WEB10K')

DATASET_BASE_PATH = '/kaggle/input/datasets/engkhaledmo/mslr-10k/'

# ===== Debug config ditulis di Notebook (bukan dari YAML) =====
DEBUG_MODE = False
DEBUG_RATIO = 1e-6   # set 0.0 untuk full data
FOLDS = [1, 2]

# Task config untuk Feature Gating (hardcoded)
# Dynamic approach: bisa berapa saja jumlah tasks
TASK_INDICES = [0, 131, 132, 133]
N_TASKS = len(TASK_INDICES)
GATING_TYPE = 'soft'  # sigmoid gating
MOO_METHOD = 'ls'  # least squares
REDUCTION_METHOD = 'mean'
LABEL_INDICES = TASK_INDICES[1:]  # Auxiliary tasks (semua kecuali task 0)

# Output
OUTPUT_DIR = os.path.join(EXAMPLE_DIR, 'outputs', 'results')
CHECKPOINT_DIR = os.path.join(EXAMPLE_DIR, 'checkpoints')

# Config JSON model gating
CONFIG_GATING = os.path.join(EXAMPLE_DIR, 'configs', 'config_gating.json')

for p in [PROJECT_ROOT, EXAMPLE_DIR, OUTPUT_DIR, CHECKPOINT_DIR]:
    if not os.path.exists(p) and p in [OUTPUT_DIR, CHECKPOINT_DIR]:
        os.makedirs(p, exist_ok=True)

print('PROJECT_ROOT:', PROJECT_ROOT)
print('EXAMPLE_DIR :', EXAMPLE_DIR)
print('DATASET_BASE_PATH:', DATASET_BASE_PATH)
print('OUTPUT_DIR:', OUTPUT_DIR)
print('DEBUG_MODE:', DEBUG_MODE, '| DEBUG_RATIO:', DEBUG_RATIO)
print(f'TASK_INDICES: {TASK_INDICES} (n_tasks={N_TASKS})')
print('LABEL_INDICES:', LABEL_INDICES)
print('GATING_TYPE:', GATING_TYPE)

In [ ]:
# Override training epochs for non-debug runs (Notebook-owned)
EPOCHS_OVERRIDE = 50
ORIG_CONFIG_PATH = os.path.join(EXAMPLE_DIR, 'configs', 'config_gating.json')
OVERRIDE_CONFIG_PATH = os.path.join(EXAMPLE_DIR, 'configs', 'config_gating_epochs50.json')

with open(ORIG_CONFIG_PATH, 'r', encoding='utf-8') as f:
    _cfg = json.load(f)

_cfg.setdefault('training', {})
_cfg['training']['epochs'] = EPOCHS_OVERRIDE - 1  # Convert to 0-based: 50 - 1 = 49

with open(OVERRIDE_CONFIG_PATH, 'w', encoding='utf-8') as f:
    json.dump(_cfg, f, indent=2)

CONFIG_GATING = OVERRIDE_CONFIG_PATH
print('Config override written to:', CONFIG_GATING)

In [ ]:
import sys
import importlib.util

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
if EXAMPLE_DIR not in sys.path:
    sys.path.insert(0, EXAMPLE_DIR)

RUN_EXTENSION_PATH = os.path.join(EXAMPLE_DIR, 'run_extension.py')
if os.path.exists(RUN_EXTENSION_PATH):
    spec = importlib.util.spec_from_file_location('run_extension', RUN_EXTENSION_PATH)
    rx = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(rx)
else:
    import run_extension as rx

# Override variabel global modul -> tidak bergantung debug config YAML
rx.DATASET_BASE_PATH = DATASET_BASE_PATH
rx.FOLDS = FOLDS
rx.REDUCTION_METHOD = REDUCTION_METHOD
rx.LABEL_INDICES = LABEL_INDICES
rx.OUTPUT_DIR = OUTPUT_DIR
rx.CHECKPOINT_DIR = CHECKPOINT_DIR
rx.CONFIG_GATING = CONFIG_GATING
rx.MOO_METHOD = MOO_METHOD
rx.DEBUG = DEBUG_MODE

# Convert task indices to string format as expected by run_extension
rx.TASK_INDICES = ','.join(map(str, TASK_INDICES))

print('run_extension loaded from:', RUN_EXTENSION_PATH)
print('run_extension globals overridden from notebook runtime config.')

In [ ]:
def summarize_metrics(agg_dict):
    """Aggregate per-fold metrics into mean±std summary."""
    out = {}
    for metric_name, values in agg_dict.items():
        arr = np.array(values, dtype=float)
        out[metric_name] = {
            'mean': float(np.mean(arr)),
            'std': float(np.std(arr, ddof=1)) if len(arr) > 1 else 0.0,
            'per_fold': [float(x) for x in arr.tolist()]
        }
    return out

def metrics_summary_to_df(summary_dict, experiment_name):
    """Convert metrics summary to DataFrame."""
    rows = []
    for m, v in summary_dict.items():
        rows.append({
            'experiment': experiment_name,
            'metric': m,
            'mean': v['mean'],
            'std': v['std'],
        })
    if not rows:
        return pd.DataFrame(columns=['experiment', 'metric', 'mean', 'std'])
    return pd.DataFrame(rows).sort_values('metric').reset_index(drop=True)

def load_weight_combinations(weights_file_path, n_tasks):
    """
    Load weight combinations dari file dan extract weights untuk n_tasks.
    
    Args:
        weights_file_path: Path ke weights file
        n_tasks: Jumlah tasks yang digunakan
        
    Returns:
        List of weight arrays (setiap array punya length = n_tasks)

    Behavior:
        - Jika file punya > n_tasks weights per line, ambil first n_tasks
        - Jika file punya < n_tasks weights per line, raise error
        
    Note: run_extension.py juga melakukan validation yang sama, ini hanya untuk
          early validation di notebook level.
    """
    weights = []
    with open(weights_file_path, 'r') as f:
        for line_num, line in enumerate(f, 1):
            line = line.strip()
            if not line:
                continue
            
            values = [float(x) for x in line.split()]
            
            if len(values) < n_tasks:
                raise ValueError(
                    f'Line {line_num}: Expected >= {n_tasks} weights, got {len(values)}\n'
                    f'File has fewer weights than n_tasks. Cannot handle this case.'
                )
            
            # Ambil first n_tasks weights (dynamic approach)
            selected_weights = values[:n_tasks]
            weights.append(selected_weights)
    
    print(f'✓ Loaded {len(weights)} weight combinations for {n_tasks} tasks')
    print(f'  First combination: {weights[0]}')
    print(f'  Last combination: {weights[-1]}')
    return weights

print('Helper functions defined.')

In [ ]:
# ==============================
# Run Feature Gating Experiments
# ==============================

# Ensure weights-5tasks.txt is available (required by run_extension.py)
weights_file = os.path.join(EXAMPLE_DIR, 'weights-5tasks.txt')
if not os.path.exists(weights_file):
    print(f'WARNING: weights-5tasks.txt not found at {weights_file}')
    print('Attempting to locate...')
else:
    print(f'✓ weights-5tasks.txt found at: {weights_file}')

# Dynamically load and validate weights for current n_tasks
print(f'\nValidating weight combinations for {N_TASKS} tasks...')
try:
    weight_combinations = load_weight_combinations(weights_file, N_TASKS)
    print(f'✓ Weight validation passed: {len(weight_combinations)} combinations available')
except ValueError as e:
    print(f'✗ Weight validation failed: {e}')
    raise

# Change to EXAMPLE_DIR so run_extension.py can find weights-5tasks.txt with relative path
original_cwd = os.getcwd()
os.chdir(EXAMPLE_DIR)
print(f'\n✓ Changed working directory to: {os.getcwd()}')

metrics_agg = defaultdict(list)
fold_details = []
all_gating_sparsity = defaultdict(list)  # Store gating sparsity per fold
all_ndcg30 = []
all_delta_m = []
all_robustness = defaultdict(list)

try:
    for fold in FOLDS:
        print('\n' + '=' * 70)
        print(f'Processing Fold {fold} - Dynamic Feature Gating')
        print(f'Tasks: {TASK_INDICES} | MOO Method: {MOO_METHOD} | Gating: {GATING_TYPE.upper()}')
        print('=' * 70)

        dataset_path = os.path.join(DATASET_BASE_PATH, f'Fold{fold}')
        if not os.path.exists(dataset_path):
            raise FileNotFoundError(f'Dataset path not found: {dataset_path}')

        # Load dataset
        max_rows = None
        if DEBUG_MODE and DEBUG_RATIO > 0:
            estimated_total_rows = 30000000
            max_rows = max(1, int(estimated_total_rows * DEBUG_RATIO))

        train_ds, val_ds = rx.load_libsvm_dataset(
            dataset_path, 200, 'vali', max_rows=max_rows
        )

        nf = train_ds.shape[-1] - len(LABEL_INDICES)
        print(f'Dataset loaded: Train shape: {train_ds.shape}, Val shape: {val_ds.shape}')
        print(f'Number of features: {nf}')

        train_dl, val_dl = rx.create_data_loaders(
            train_ds, val_ds, 4, 64
        )

        # Run gating experiment using run_extension module
        try:
            # Call the main experiment function from run_extension
            result = rx.run_experiment(
                experiment_name=f'Gating-Fold{fold}',
                config_path=CONFIG_GATING,
                dataset_path=dataset_path,
                train_dataloader=train_dl,
                val_dataloader=val_dl,
                n_features=nf,
                use_mrl=False,
                use_gating=True,
                task_indices=TASK_INDICES
            )

            # Find the best result among all weight combinations based on task 0 NDCG@30
            best_weight = -1
            best_ndcg = -1.0
            best_res = {}
            
            for w_idx, res in result.items():
                ndcg = res.get('val_metrics', {}).get(0, {}).get('ndcg_30', 0.0)
                if ndcg > best_ndcg:
                    best_ndcg = ndcg
                    best_weight = w_idx
                    best_res = res
                    
            print(f"Selected best weight index {best_weight} with NDCG@30: {best_ndcg}")

            # Extract metrics from the best result
            fold_metrics = best_res.get('val_metrics', {})
            for task_idx, task_metrics in fold_metrics.items():
                for metric_name, metric_value in task_metrics.items():
                    if not isinstance(metric_value, dict):
                        key = f'task{task_idx}_{metric_name}'
                        metrics_agg[key].append(float(metric_value))

            # Extract special metrics (gating sparsity)
            special_metrics = best_res.get('special_metrics', {})
            for metric_name, metric_value in special_metrics.items():
                if metric_name == 'gating_sparsity_ratio':
                    all_gating_sparsity[fold] = metric_value
                if not isinstance(metric_value, dict):
                    metrics_agg[metric_name].append(float(metric_value))

            # Extract main task NDCG@30 and delta_m
            ndcg30 = fold_metrics.get(0, {}).get('ndcg_30', 0.0)
            all_ndcg30.append(float(ndcg30))

            delta_m = best_res.get('delta_m_percent', 0.0)
            all_delta_m.append(float(delta_m))

            # Robustness metrics (if available)
            if 'noise_robustness' in special_metrics:
                for t_idx, rob in special_metrics['noise_robustness'].items():
                    for k, v in rob.items():
                        all_robustness[f"task{t_idx}_{k}"].append(float(v))

            fold_details.append({
                'fold': fold,
                'metrics': fold_metrics,
                'special_metrics': special_metrics,
                'num_params': best_res.get('num_params', 0),
                'delta_m_percent': delta_m,
                'gating_sparsity': special_metrics.get('gating_sparsity_ratio', None)
            })

        except Exception as e:
            print(f'✗ Error running experiment for fold {fold}: {e}')
            raise

        del train_ds, val_ds, train_dl, val_dl
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        gc.collect()

    print('\n' + '=' * 70)
    print('✓ All folds completed successfully!')
    print('=' * 70)

finally:
    # Restore original working directory
    os.chdir(original_cwd)
    print(f'✓ Restored working directory to: {os.getcwd()}')

In [ ]:
# Aggregate results
summary = summarize_metrics(metrics_agg)

# Compute final delta_m and robustness averages
avg_delta_m = float(np.mean(all_delta_m)) if all_delta_m else 0.0
avg_ndcg30 = float(np.mean(all_ndcg30)) if all_ndcg30 else 0.0

# Compute gating sparsity average
all_gating_sparsity_values = list(all_gating_sparsity.values())
avg_gating_sparsity = float(np.mean(all_gating_sparsity_values)) if all_gating_sparsity_values else 0.0

robustness_summary = {}
for k, v_list in all_robustness.items():
    arr = np.array(v_list, dtype=float)
    robustness_summary[k] = {
        'mean': float(np.mean(arr)),
        'std': float(np.std(arr, ddof=1)) if len(arr) > 1 else 0.0,
        'per_fold': [float(x) for x in arr.tolist()]
    }

summary_out = {
    'method': 'Dynamic Feature Gating',
    'folds': FOLDS,
    'task_indices': TASK_INDICES,
    'gating_type': GATING_TYPE,
    'debug_mode': DEBUG_MODE,
    'debug_ratio': DEBUG_RATIO,
    'ndcg30_avg': avg_ndcg30,
    'ndcg30_per_fold': all_ndcg30,
    'delta_m_percent_avg': avg_delta_m,
    'delta_m_percent_per_fold': all_delta_m,
    'gating_sparsity_avg': avg_gating_sparsity,
    'gating_sparsity_per_fold': all_gating_sparsity_values,
    'metrics': summary,
    'robustness': robustness_summary,
    'fold_details': fold_details
}

gating_dir = os.path.join(OUTPUT_DIR, 'gating')
os.makedirs(gating_dir, exist_ok=True)
summary_path = os.path.join(gating_dir, 'gating_summary.json')
with open(summary_path, 'w', encoding='utf-8') as f:
    json.dump(summary_out, f, indent=2, default=float)

print('Gating summary saved to:', summary_path)
print(f'\nFeature Gating NDCG@30 avg: {avg_ndcg30:.6f}')
print(f'Gating Sparsity Ratio avg: {avg_gating_sparsity:.6f}')
print(f'Delta m%% avg: {avg_delta_m:+.4f}%')
print(f'\nMetrics aggregated across {len(FOLDS)} folds:')
for metric_name in sorted(summary.keys()):
    if not metric_name.startswith('task'):
        metric_stats = summary[metric_name]
        print(f"  {metric_name}: {metric_stats['mean']:.4f} ± {metric_stats['std']:.4f}")

In [ ]:
# Tampilkan semua metrik agregat (mean ± std)
df = metrics_summary_to_df(summary, 'gating')
display(df.sort_values('metric').reset_index(drop=True))

print('\nKey Metrics Summary (Feature Gating-Specific):')
print(f"  NDCG@30 (Main Task): {summary.get('task0_ndcg_30', {}).get('mean', 0):.4f}")
print(f"  Gating Sparsity Ratio: {avg_gating_sparsity:.6f}")
print(f"  Delta m%: {avg_delta_m:+.4f}%")
print(f"  Robustness metrics: {list(robustness_summary.keys())}")

In [ ]:
# Plot 1: NDCG metrics per fold
ndcg_metrics = sorted([m for m in summary.keys() 
                        if 'ndcg_' in m and not m.startswith('task')],
                       key=lambda x: int(x.split('_')[1]))

if ndcg_metrics:
    ncols = 3
    nrows = int(np.ceil(len(ndcg_metrics) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 3.5 * nrows), squeeze=False)

    for i, m in enumerate(ndcg_metrics):
        r, c = divmod(i, ncols)
        ax = axes[r][c]
        per_fold = summary[m]['per_fold']
        ax.plot(FOLDS, per_fold, marker='o', linewidth=2, markersize=8, color='tab:purple')
        ax.set_title(f'{m.upper()} (Main Task)', fontsize=11, fontweight='bold')
        ax.set_xlabel('Fold')
        ax.set_ylabel('Score')
        ax.set_ylim([0, 1])
        ax.grid(True, alpha=0.3)

    # Hide unused axes
    total_axes = nrows * ncols
    for j in range(len(ndcg_metrics), total_axes):
        r, c = divmod(j, ncols)
        axes[r][c].axis('off')

    fig.suptitle('NDCG@k Scores by Fold - Feature Gating', y=1.00, fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()
else:
    print('No NDCG metrics found to plot.')

In [ ]:
# Plot 2: Gating Sparsity Ratio (Feature Gating-specific metric)
if all_gating_sparsity:
    fig, ax = plt.subplots(figsize=(8, 5))
    
    sparsity_values = [all_gating_sparsity[fold] for fold in FOLDS]
    bars = ax.bar(['Sparsity (Mean)', *[f'Fold{f}' for f in FOLDS]], 
                   [avg_gating_sparsity, *sparsity_values],
                   color=['darkred'] + ['salmon'] * len(FOLDS),
                   edgecolor='black', linewidth=1.5)
    
    # Add value labels on bars
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.6f}',
                ha='center', va='bottom', fontweight='bold')
    
    ax.set_ylabel('Sparsity Ratio', fontsize=11)
    ax.set_title('Gating Sparsity Ratio by Fold\n(Persentase fitur dengan bobot mendekati nol)', 
                 fontsize=13, fontweight='bold')

    max_val = max(sparsity_values + [avg_gating_sparsity])
    ax.set_ylim([0, max_val * 1.2 if max_val > 0 else 1.0])
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print('No gating sparsity data found.')

In [ ]:
# Plot 3: Delta m% (Average Relative Improvement)
fig, ax = plt.subplots(figsize=(8, 5))
color = 'green' if avg_delta_m >= 0 else 'red'
bars = ax.bar(['Δm% (Mean)', *[f'Fold{f}' for f in FOLDS]], 
               [avg_delta_m, *all_delta_m],
               color=[color] + ['lightblue'] * len(FOLDS),
               edgecolor='black', linewidth=1.5)

# Add value labels on bars
for bar in bars:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{height:.2f}%',
            ha='center', va='bottom', fontweight='bold')

ax.axhline(0, color='black', linewidth=1)
ax.set_ylabel('Percent (%)', fontsize=11)
ax.set_title('Average Relative Improvement vs Single-Task (Δm%)', 
             fontsize=13, fontweight='bold')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Plot 4: Per-Task NDCG@30 comparison
task_metrics = {}
for m in summary.keys():
    if 'task' in m and 'ndcg_30' in m:
        task_id = int(m.split('_')[0].replace('task', ''))
        task_metrics[task_id] = summary[m]['mean']

if task_metrics:
    fig, ax = plt.subplots(figsize=(10, 5))
    task_ids = sorted(task_metrics.keys())
    colors = ['tab:blue'] + ['tab:orange'] * (len(task_ids) - 1)
    bars = ax.bar([str(tid) for tid in task_ids], 
                   [task_metrics[tid] for tid in task_ids],
                   color=colors, edgecolor='black', linewidth=1.5)

    # Add value labels
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.4f}',
                ha='center', va='bottom', fontsize=9)

    ax.set_xlabel('Task ID', fontsize=11)
    ax.set_ylabel('NDCG@30', fontsize=11)
    ax.set_title('Per-Task Performance (NDCG@30) - Feature Gating',
                 fontsize=13, fontweight='bold')
    ax.set_ylim([0, 1])
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.show()

In [ ]:
# Plot 5: Robustness to Noisy Features (Feature Gating-specific metric)
if robustness_summary:
    fig, ax = plt.subplots(figsize=(10, 5))
    
    robustness_names = sorted(robustness_summary.keys())
    means = [robustness_summary[name]['mean'] for name in robustness_names]
    stds = [robustness_summary[name]['std'] for name in robustness_names]
    
    x = np.arange(len(robustness_names))
    bars = ax.bar(x, means, yerr=stds, capsize=5, 
                  color='teal', edgecolor='black', linewidth=1.5, alpha=0.7)
    
    # Add value labels
    for bar, mean, std in zip(bars, means, stds):
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{mean:.2%}\n±{std:.2%}',
                ha='center', va='bottom', fontsize=9)
    
    ax.set_xlabel('Robustness Metric', fontsize=11)
    ax.set_ylabel('Value', fontsize=11)
    ax.set_title('Robustness to Noisy Features - Feature Gating\n(Ketahanan model terhadap gangguan fitur)',
                fontsize=13, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(robustness_names, rotation=45, ha='right')
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print('No robustness metrics data found.')

In [ ]:
# Plot 6: Comparison of Gating Sparsity and Robustness Across Folds
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Sparsity trend
sparsity_per_fold = [all_gating_sparsity[fold] for fold in FOLDS]
ax1.plot(FOLDS, sparsity_per_fold, marker='s', linewidth=2.5, markersize=10, 
         color='red', label='Gating Sparsity')
ax1.set_xlabel('Fold', fontsize=11)
ax1.set_ylabel('Sparsity Ratio', fontsize=11)
ax1.set_title('Gating Sparsity Ratio Trend Across Folds', fontsize=12, fontweight='bold')
ax1.grid(True, alpha=0.3)
ax1.legend()

# Robustness trend for main metric
if robustness_summary:
    first_robustness_metric = sorted(robustness_summary.keys())[0]
    robustness_per_fold = robustness_summary[first_robustness_metric]['per_fold']
    ax2.plot(FOLDS, robustness_per_fold, marker='^', linewidth=2.5, markersize=10,
             color='teal', label=first_robustness_metric)
    ax2.set_xlabel('Fold', fontsize=11)
    ax2.set_ylabel('Robustness Value', fontsize=11)
    ax2.set_title(f'Robustness Trend: {first_robustness_metric}', fontsize=12, fontweight='bold')
    ax2.grid(True, alpha=0.3)
    ax2.legend()

fig.suptitle('Feature Gating - Special Metrics Trends Across Folds', 
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Buat tabel ringkasan komprehensif
print('\n' + '=' * 80)
print('DYNAMIC FEATURE GATING - COMPREHENSIVE SUMMARY')
print('=' * 80)

# Main metrics table
main_metrics_data = {
    'Metric': ['NDCG@30 (Main)', 'Gating Sparsity Ratio', 'Δm% (Mean Improvement)', 'Number of Folds', 'Task Indices', 'Gating Type'],
    'Value': [
        f'{avg_ndcg30:.6f}',
        f'{avg_gating_sparsity:.6f}',
        f'{avg_delta_m:+.4f}%',
        str(len(FOLDS)),
        str(TASK_INDICES),
        GATING_TYPE.upper()
    ]
}

df_main = pd.DataFrame(main_metrics_data)
print('\n--- MAIN METRICS ---')
display(df_main)

# Per-fold breakdown
print('\n--- PER-FOLD BREAKDOWN ---')
fold_breakdown = {
    'Fold': FOLDS,
    'NDCG@30': all_ndcg30,
    'Gating Sparsity': sparsity_per_fold,
    'Δm%': all_delta_m
}
df_fold = pd.DataFrame(fold_breakdown)
display(df_fold)

# Gating-specific parameters
print('\n--- FEATURE GATING-SPECIFIC PARAMETERS ---')
gating_params = {
    'Parameter': ['Gating Type', 'MOO Method', 'Reduction Method'],
    'Value': [GATING_TYPE, MOO_METHOD, REDUCTION_METHOD]
}
df_gating = pd.DataFrame(gating_params)
display(df_gating)

# Robustness summary
if robustness_summary:
    print('\n--- ROBUSTNESS TO NOISY FEATURES ---')
    robustness_data = []
    for metric_name, values in robustness_summary.items():
        robustness_data.append({
            'Metric': metric_name,
            'Mean': f"{values['mean']:.6f}",
            'Std': f"{values['std']:.6f}",
            'Per-Fold': str(values['per_fold'])
        })
    df_robustness = pd.DataFrame(robustness_data)
    display(df_robustness)

In [ ]:
# Zip checkpoint dan result agar mudah diunduh dari Kaggle
import os
import shutil

CHECKPOINTS_ROOT = os.path.join(CHECKPOINT_DIR, 'feature_gating')
RESULTS_ROOT = gating_dir
ZIP_DIR = os.path.join(PROJECT_ROOT, 'archives')
os.makedirs(ZIP_DIR, exist_ok=True)

checkpoint_zip_base = os.path.join(ZIP_DIR, 'gating_checkpoints')
results_zip_base = os.path.join(ZIP_DIR, 'gating_results')

if os.path.exists(CHECKPOINTS_ROOT):
    checkpoint_zip_path = shutil.make_archive(checkpoint_zip_base, 'zip', CHECKPOINTS_ROOT)
    print('Checkpoint ZIP:', checkpoint_zip_path)
else:
    print('Checkpoint folder not found:', CHECKPOINTS_ROOT)

if os.path.exists(RESULTS_ROOT):
    results_zip_path = shutil.make_archive(results_zip_base, 'zip', RESULTS_ROOT)
    print('Results ZIP:', results_zip_path)
else:
    print('Results folder not found:', RESULTS_ROOT)

print('Archive directory:', ZIP_DIR)

## Catatan - Dynamic Configuration (End-to-End)

### Arsitektur Solusi Dinamis

Notebook dan `run_extension.py` bekerja sama untuk support arbitrary jumlah tasks:

**Level 1: Notebook (Early Validation)**
- Membaca TASK_INDICES dan menghitung N_TASKS
- `load_weight_combinations()` memvalidasi weights file sebelum training
- Print: jumlah weight combinations, first/last combination
- Jika ada error, fail early sebelum GPU computation

**Level 2: run_extension.py (Dynamic Weight Handling)**
- `get_weight_combinations(moo_method, num_tasks, task_indices)` sekarang **fully dynamic**
- Support hardcoded cases:
  - `num_tasks == 2`: Generate linear interpolation weights
  - `num_tasks == 6`: Load weights-5tasks.txt dan prepend uniform weight untuk task 0
- Support arbitrary cases:
  - `num_tasks != 2 and != 6`: Load weights-5tasks.txt dan ambil **first N_TASKS columns**
  - Validation: raise error jika file tidak punya cukup columns

### Pendekatan Dinamis - Cara Kerja

1. **Set TASK_INDICES sekali**, sisanya otomatis:
```python
TASK_INDICES = [0, 131, 132, 133, 134, 135]  # 6 tasks
# Otomatis:
# - N_TASKS = 6
# - LABEL_INDICES = [131, 132, 133, 134, 135]
# - Notebook: validate weights untuk 6 tasks
# - run_extension.py: ambil first 6 weights dari weights-5tasks.txt
```

2. **Mau ubah ke 4 tasks?** Hanya ubah 1 line:
```python
TASK_INDICES = [0, 131, 132, 133]  # Semuanya adjust otomatis!
```

3. **Validation Otomatis (Dual-Level):**
```
Notebook Level:
✓ weights-5tasks.txt found
✓ Validating weight combinations for 6 tasks...
✓ Loaded 80 weight combinations for 6 tasks
✓ First combination: [0.2, 0.2, 0.2, 0.2, 0.2, 0.2]
✓ Last combination: [0.61, 0.06, 0.06, 0.05, 0.11, 0.11]

run_extension.py Level:
✓ Weight combinations determined: 80 combinations for 6 tasks
✓ Starting training: Gating-Fold1 (weight 0)...
```

### Metrik Khusus Feature Gating (Fase 1)

Berdasarkan `metrics.md`:

- **Gating Sparsity Ratio**: Persentase fitur input yang diberikan bobot mendekati nol oleh mekanisme Gating. Menunjukkan tingkat efisiensi pembuangan *noise*.
- **Robustness to Noisy Features**: Persentase penurunan skor NDCG ketika sebagian fitur sengaja diberi gangguan (*noise*). Menguji apakah Gating secara eksplisit mampu "menutup pintu" bagi fitur buruk.
- **Δm%**: Average relative improvement vs single-task baseline
- **NDCG@1, @5, @10, @20, @30**: Metrik ranking utama
- **MAP & MRR**: Metrik akurasi dokumen relevan

### Gating Parameters

- Gating type: `soft` (Sigmoid)
- MOO method: `ls` (least squares)
- Reduction method: `mean`

### Tips Penggunaan

- Untuk 4 tasks: `TASK_INDICES = [0, 131, 132, 133]` ✅
- Untuk 6 tasks: `TASK_INDICES = [0, 131, 132, 133, 134, 135]` ✅
- Untuk N tasks: Set `TASK_INDICES` dengan N elements, dan semua handle otomatis
- Persyaratan: `weights-5tasks.txt` harus punya ≥ N_TASKS columns
- **Peringatan**: Pastikan `run_extension.py` sudah di-update untuk mendukung Feature Gating mode (`use_gating=True`)

### Perbandingan dengan Matryoshka

| Aspek | Matryoshka | Feature Gating |
|-------|-----------|----------------|
| **Metrik Khusus** | Effective Dimensionality Efficiency | Gating Sparsity Ratio |
| **Mekanisme** | Nested dimensions [32, 64, 128, 256] | Soft sigmoid gating |
| **Focus** | Information density di berbagai dimensi | Feature selection & noise filtering |
| **Robustness** | Fleksibilitas alami representasi padat | Eksplisit feature suppression |
| **Checkpoint Path** | `checkpoints/matryoshka/` | `checkpoints/feature_gating/` |
| **Config** | `config_matryoshka.json` | `config_gating.json` |